# Keys and Constraints

## Overview
**Constraints** are rules enforced by the database engine that guarantee data integrity. They are the last line of defence -- application bugs, ETL errors, and manual inserts cannot violate them.

**PostgreSQL constraint types:**

| Constraint | Purpose | Example |
|---|---|---|
| `PRIMARY KEY` | Uniquely identifies each row; implies UNIQUE + NOT NULL | `patient_id SERIAL PRIMARY KEY` |
| `FOREIGN KEY` | Enforces referential integrity between tables | `REFERENCES patients(patient_id)` |
| `UNIQUE` | No two rows can have the same value(s) | `UNIQUE (licence_no)` |
| `NOT NULL` | Column cannot be NULL | `last_name TEXT NOT NULL` |
| `CHECK` | Value must satisfy a condition | `CHECK(sex IN ('M','F','O'))` |
| `DEFAULT` | Value used when none is supplied | `DEFAULT 1` |
| `EXCLUDE` | No two rows have conflicting values (PostgreSQL only) | Overlapping appointment slots |

**Referential integrity actions (ON DELETE / ON UPDATE):**

| Action | Behaviour |
|---|---|
| `RESTRICT` / `NO ACTION` | Prevents deletion/update if referenced rows exist (default) |
| `CASCADE` | Automatically deletes/updates referencing rows |
| `SET NULL` | Sets FK column to NULL when parent is deleted |
| `SET DEFAULT` | Sets FK column to its default when parent is deleted |

**SQLite note:** SQLite parses FK constraint syntax but does NOT enforce foreign keys by default. Run `PRAGMA foreign_keys = ON` in every connection to enable enforcement. PostgreSQL enforces FKs by default.

---

In [ ]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE patients (patient_id INTEGER PRIMARY KEY, first_name TEXT NOT NULL, last_name TEXT NOT NULL, dob TEXT NOT NULL, sex TEXT CHECK(sex IN ('M','F','O')), province TEXT, active INTEGER DEFAULT 1);CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT NOT NULL UNIQUE, building TEXT, floor_no INTEGER);CREATE TABLE providers (provider_id INTEGER PRIMARY KEY, full_name TEXT NOT NULL, specialty TEXT, licence_no TEXT UNIQUE, province TEXT, dept_id INTEGER REFERENCES departments(dept_id), manager_id INTEGER REFERENCES providers(provider_id));CREATE TABLE diagnoses (diag_code TEXT PRIMARY KEY, description TEXT NOT NULL, category TEXT NOT NULL);CREATE TABLE encounters (enc_id INTEGER PRIMARY KEY, patient_id INTEGER NOT NULL REFERENCES patients(patient_id), provider_id INTEGER NOT NULL REFERENCES providers(provider_id), dept_id INTEGER REFERENCES departments(dept_id), enc_date TEXT NOT NULL, cost_cad REAL, admitted INTEGER DEFAULT 0 CHECK(admitted IN (0,1)));CREATE TABLE encounter_diagnoses (enc_id INTEGER NOT NULL REFERENCES encounters(enc_id), diag_code TEXT NOT NULL REFERENCES diagnoses(diag_code), diag_rank INTEGER DEFAULT 1, confirmed INTEGER DEFAULT 1, PRIMARY KEY (enc_id, diag_code));CREATE TABLE lab_results (result_id INTEGER PRIMARY KEY, patient_id INTEGER NOT NULL REFERENCES patients(patient_id), test_name TEXT NOT NULL, result_val REAL, units TEXT, collected TEXT NOT NULL, flagged INTEGER DEFAULT 0);INSERT INTO departments VALUES (1,'Emergency','Tower A',1),(2,'Cardiology','Tower B',2),(3,'Oncology','Tower C',3),(4,'Family Medicine','Clinic',1),(5,'Orthopaedics','Tower A',2),(6,'Radiology','Tower B',3),(7,'ICU','Tower C',NULL);INSERT INTO providers VALUES (10,'Dr. Sarah MacNeil','Cardiology','MC-1001','NB',2,NULL),(11,'Dr. James Wong','Oncology','MC-1002','BC',3,10),(12,'Dr. Fatima Osei','Family Medicine','MC-1003','ON',4,10),(13,'Dr. Ethan Larson','Orthopaedics','MC-1004','QC',5,10),(14,'Dr. Priya Sharma','Emergency','MC-1005','AB',1,10),(15,'Dr. Lucas Petit','Radiology','MC-1006','QC',6,11);INSERT INTO patients VALUES (1,'Aroha','Ngata','1985-03-12','F','NB',1),(2,'Liam','Chen','1972-11-04','M','NS',1),(3,'Fatima','Al-Rashid','1990-07-22','F','ON',1),(4,'James','MacLeod','1955-01-30','M','NB',0),(5,'Sofia','Petrov','2001-09-15','F','BC',1),(6,'Noah','Williams','1968-06-08','M','AB',1),(7,'Mei','Zhang','1995-04-17','F','ON',1),(8,'Ethan','Tremblay','1980-12-01','M','QC',0),(9,'Priya','Nair','1977-08-25','F','BC',1),(10,'Marcus','Okafor','1993-05-19','M','ON',1),(11,'Diana','Flores','2000-02-14','F','NB',1);INSERT INTO diagnoses VALUES ('I25.1','Atherosclerotic heart disease','Cardiovascular'),('J18.9','Pneumonia, unspecified','Respiratory'),('Z00.0','General medical examination','Preventive'),('M17.1','Primary osteoarthritis of knee','Musculoskeletal'),('J06.9','Acute upper respiratory infection','Respiratory'),('R07.9','Chest pain, unspecified','Symptoms'),('I10','Essential hypertension','Cardiovascular'),('R55','Syncope and collapse','Symptoms'),('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),('S52.5','Fracture of lower end of radius','Injury'),('E11.9','Type 2 diabetes mellitus','Endocrine'),('I50.9','Heart failure, unspecified','Cardiovascular');INSERT INTO encounters VALUES (1,1,10,2,'2023-01-15',3200.00,1),(2,2,14,1,'2023-02-20',1850.00,1),(3,3,12,4,'2023-03-05',120.00,0),(4,4,13,5,'2023-03-18',5500.00,1),(5,5,12,4,'2023-04-02',95.00,0),(6,6,14,1,'2023-05-11',780.00,0),(7,7,10,2,'2023-06-22',2100.00,1),(8,8,12,4,'2023-07-14',80.00,0),(9,1,14,1,'2023-08-30',410.00,0),(10,3,12,4,'2023-09-12',110.00,0),(11,9,10,2,'2023-10-01',1750.00,1),(12,10,14,1,'2023-11-03',2200.00,1),(13,2,10,2,'2023-11-20',2900.00,1),(14,6,12,4,'2023-12-01',115.00,0);INSERT INTO encounter_diagnoses VALUES (1,'I25.1',1,1),(1,'I10',2,1),(2,'J18.9',1,1),(3,'Z00.0',1,1),(4,'M17.1',1,1),(5,'J06.9',1,1),(6,'R07.9',1,1),(7,'I10',1,1),(7,'I48.0',2,1),(9,'R55',1,1),(10,'Z00.0',1,1),(11,'I48.0',1,1),(12,'S52.5',1,1),(13,'I25.1',1,1),(14,'Z00.0',1,1);INSERT INTO lab_results VALUES (1,1,'HbA1c',7.2,'%','2023-03-10',0),(2,2,'HbA1c',9.1,'%','2023-03-15',1),(3,3,'Creatinine',88.5,'umol/L','2023-04-01',0),(4,4,'Creatinine',145.0,'umol/L','2023-04-12',1),(5,5,'HbA1c',5.8,'%','2023-05-05',0),(6,6,'Sodium',138.0,'mmol/L','2023-05-20',0),(7,7,'Sodium',151.0,'mmol/L','2023-06-01',1),(8,1,'Creatinine',72.3,'umol/L','2023-06-18',0),(9,8,'HbA1c',6.4,'%','2023-07-02',0),(10,9,'Creatinine',210.0,'umol/L','2023-07-15',1),(11,2,'Creatinine',95.0,'umol/L','2023-08-01',0),(12,10,'HbA1c',7.8,'%','2023-08-20',1);""")
conn.commit()
print("Healthcare schema ready.")
for t in ["patients","providers","departments","diagnoses","encounters","encounter_diagnoses","lab_results"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} rows")
conn.execute("PRAGMA foreign_keys = ON")
print("Foreign key enforcement enabled.")

---
## PRIMARY KEY -- uniqueness and NOT NULL

In [ ]:
# Demonstrate PK enforcement
print("=== PRIMARY KEY enforces uniqueness ===")
try:
    conn.execute("INSERT INTO patients VALUES (1,'Duplicate','Patient','2000-01-01','F','ON',1)")
    conn.commit()
    print("ERROR: duplicate patient_id=1 was allowed (should not happen)")
except Exception as e:
    print(f"Correctly rejected duplicate PK: {e}")

print()
print("=== Composite PRIMARY KEY in bridge table ===")
print("encounter_diagnoses has PRIMARY KEY (enc_id, diag_code)")
print("Same enc_id can appear multiple times (different diag_codes)")
print("Same diag_code can appear multiple times (different enc_ids)")
print("But the combination (enc_id, diag_code) must be unique")
try:
    conn.execute("INSERT INTO encounter_diagnoses VALUES (1,'I25.1',3,1)")
    conn.commit()
    print("ERROR: duplicate (enc_id=1, diag_code='I25.1') allowed (should not happen)")
except Exception as e:
    print(f"Correctly rejected duplicate composite PK: {e}")
print()
q = "SELECT enc_id, diag_code, diag_rank FROM encounter_diagnoses WHERE enc_id=1"
print("Existing rows for enc_id=1:")
print(pd.read_sql(q, conn).to_string(index=False))


---
## FOREIGN KEY and referential integrity

In [ ]:
# FK prevents orphaned rows
print("=== FOREIGN KEY prevents orphaned references ===")
# Try to insert an encounter for a non-existent patient
try:
    conn.execute("INSERT INTO encounters VALUES (99,999,10,2,'2023-12-15',500.00,0)")
    conn.commit()
    print("ERROR: orphaned patient_id=999 was allowed")
except Exception as e:
    print(f"Correctly rejected missing patient FK: {e}")

# ON DELETE CASCADE vs RESTRICT
print()
print("=== ON DELETE behaviour ===")
print("Our schema uses RESTRICT (default): cannot delete a patient with encounters.")
try:
    conn.execute("DELETE FROM patients WHERE patient_id = 1")
    conn.commit()
    print("ERROR: patient with encounters was deleted")
except Exception as e:
    print(f"Correctly blocked delete of patient with encounters: {e}")

print()
print("PostgreSQL CASCADE example (not testable in this SQLite session):")
print("""
  -- Patient deletion cascades to encounters and lab results:
  FOREIGN KEY (patient_id) REFERENCES patients(patient_id) ON DELETE CASCADE

  -- Provider removal sets dept_id to NULL on remaining providers:
  FOREIGN KEY (dept_id) REFERENCES departments(dept_id) ON DELETE SET NULL
""")


---
## UNIQUE, CHECK, and NOT NULL

In [ ]:
print("=== UNIQUE constraint: no two providers share a licence number ===")
try:
    conn.execute("INSERT INTO providers VALUES (99,'Dr. Fake','Cardiology','MC-1001','ON',2,NULL)")
    conn.commit()
    print("ERROR: duplicate licence_no allowed")
except Exception as e:
    print(f"Correctly rejected duplicate licence_no: {e}")

print()
print("=== CHECK constraint: sex must be M, F, or O ===")
try:
    conn.execute("INSERT INTO patients VALUES (99,'Bad','Data','1990-01-01','X','ON',1)")
    conn.commit()
    print("ERROR: invalid sex value allowed")
except Exception as e:
    print(f"Correctly rejected invalid sex value: {e}")

print()
print("=== NOT NULL: first_name cannot be NULL ===")
try:
    conn.execute("INSERT INTO patients (patient_id,last_name,dob) VALUES (99,'NoFirst','1990-01-01')")
    conn.commit()
    print("ERROR: NULL first_name allowed")
except Exception as e:
    print(f"Correctly rejected NULL first_name: {e}")

print()
print("=== Adding constraints to an existing table (PostgreSQL) ===")
print("""
  -- Add a CHECK constraint after table creation:
  ALTER TABLE encounters
      ADD CONSTRAINT chk_cost_positive CHECK (cost_cad >= 0);

  -- Add a NOT NULL constraint:
  ALTER TABLE encounters
      ALTER COLUMN enc_date SET NOT NULL;

  -- Add a UNIQUE constraint:
  ALTER TABLE providers
      ADD CONSTRAINT uq_licence UNIQUE (licence_no);

  -- SQLite: cannot add constraints to existing tables via ALTER TABLE.
  -- Must recreate the table and copy data.
""")
conn.close()


---
## Common Pitfalls

**1. Not enabling foreign keys in SQLite**
`PRAGMA foreign_keys = ON` must be executed on every new SQLite connection. Without it, FK constraints are silently ignored -- you can insert orphaned rows with no error. PostgreSQL enforces FKs by default; SQLite does not.

**2. Using CASCADE delete without fully understanding the cascade chain**
`ON DELETE CASCADE` on `encounters` means deleting a patient deletes all their encounters. If `encounter_diagnoses` also has `ON DELETE CASCADE` on enc_id, that row is also deleted. Cascades can silently wipe large amounts of data. Audit the full cascade chain before enabling it in production.

**3. CHECK constraints are not enforced by older SQLite versions**
SQLite before 3.25 parses CHECK constraints but does not enforce them. Always test that your constraints actually reject bad data. PostgreSQL enforces CHECK constraints reliably.

**4. NOT NULL constraints on columns that legitimately need NULL**
`dept_id NOT NULL` on providers means a provider cannot exist without a department. If providers can be unassigned, make the FK nullable. Overly strict NOT NULL causes insert failures for legitimate data and forces workarounds like sentinel values (dept_id = 0).

**5. Adding a NOT NULL column to a large table with existing rows**
`ALTER TABLE encounters ADD COLUMN discharge_date TEXT NOT NULL` fails in PostgreSQL if existing rows have no value for the new column. Always provide a DEFAULT value or add the column as nullable and backfill before adding the NOT NULL constraint.

**6. Unique constraints on nullable columns behave unexpectedly**
In PostgreSQL (and SQL standard), `NULL != NULL` -- so a UNIQUE column can contain multiple NULL values. `UNIQUE (licence_no)` allows many providers to have NULL licences. If you need one-null-or-unique behaviour, use a partial unique index: `CREATE UNIQUE INDEX ON providers (licence_no) WHERE licence_no IS NOT NULL`.


---
*sql_methods_library - Samantha McGarrigle*